# Federated Learning for Colon Cancer Histopathology Images with PIDL

This notebook runs the **existing ResNet-18 + PIDL + federated learning pipeline** on the **colon cancer** subset of the Kaggle dataset `andrewmvd/lung-and-colon-cancer-histopathological-images`.

- The **model, loss, and FL loop** are exactly the same as in the brain tumor project.
- We only change **which dataset path we use** and **how many clients** we simulate.
- Results (metrics, timings, confusion matrix, summary, and final model weights) are written into **dataset-specific directories**, so brain, colon, and later lung runs do not mix.


In [ ]:
# Mount Google Drive (optional, e.g., if you want to copy artifacts to Drive)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone the repository (use your GitHub URL) and go into project root
# If you already have the repo under /content, skip clone and set PROJECT_DIR accordingly.
import os
REPO_URL = "https://github.com/PulockDas/brain-tumor-classification-PIDL-FL.git"  # ← set your repo URL if different
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}


In [ ]:
# Install project and dependencies (including kagglehub for dataset download)
!cd {PROJECT_DIR} && pip install -e . kagglehub


## Download colon cancer dataset from Kaggle


In [ ]:
import os
import kagglehub

# Download or reuse cached LC25000 lung & colon dataset
kaggle_path = kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images")
print("Kaggle dataset base path:", kaggle_path)

# Expected structure inside kaggle_path:
# lung_colon_image_set/
#   colon_image_sets/
#       colon_aca/
#       colon_n/
#   lung_image_sets/
#       lung_aca/
#       lung_scc/
#       lung_n/

colon_root = os.path.join(kaggle_path, "lung_colon_image_set", "colon_image_sets")
print("Colon dataset root:", colon_root)

if not os.path.isdir(colon_root):
    raise FileNotFoundError(f"Colon dataset root not found: {colon_root}")

print("Colon classes:", os.listdir(colon_root))


## Configuration


In [ ]:
import os

# Paths
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"
# Base directory for logs outside the repo; we will later copy *only* important files back into the repo.
LOG_BASE_DIR = "/content/results_colon"

# Dataset configuration
DATA_ROOT = colon_root           # Folder containing class subfolders (colon_aca, colon_n)
DATASET_NAME = "colon_cancer"  # Used only for naming result directories

# Federated learning configuration
NUM_ROUNDS = 10                  # Number of FL rounds (change as needed)
LOCAL_EPOCHS = 5                 # Local epochs per round
NUM_CLIENTS = 3                  # e.g. 3, 5, 10, ...

# Tag to distinguish multiple runs with the same setup
EXPERIMENT_TAG = f"{DATASET_NAME}_{NUM_CLIENTS}clients_{NUM_ROUNDS}rds"

# This is where train_fl.py will actually write logs and the final model,
# given how it constructs dataset- and client-specific log directories.
RUN_LOG_DIR = os.path.join(
    LOG_BASE_DIR, DATASET_NAME, f"{NUM_CLIENTS}_clients", EXPERIMENT_TAG
)

print("DATA_ROOT     :", DATA_ROOT)
print("LOG_BASE_DIR  :", LOG_BASE_DIR)
print("RUN_LOG_DIR   :", RUN_LOG_DIR)
print("DATASET_NAME  :", DATASET_NAME)
print("NUM_ROUNDS    :", NUM_ROUNDS)
print("LOCAL_EPOCHS  :", LOCAL_EPOCHS)
print("NUM_CLIENTS   :", NUM_CLIENTS)
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)


## Run Federated Learning on Colon Cancer Dataset


In [ ]:
import os
import subprocess

os.makedirs(LOG_BASE_DIR, exist_ok=True)

cmd = [
    "python", "train_fl.py",
    "--data-root", DATA_ROOT,
    "--dataset-name", DATASET_NAME,
    "--num-clients", str(NUM_CLIENTS),
    "--num-rounds", str(NUM_ROUNDS),
    "--local-epochs", str(LOCAL_EPOCHS),
    "--log-dir", LOG_BASE_DIR,
    "--experiment-tag", EXPERIMENT_TAG,
]
print("Running command:\n", " ".join(cmd))
result = subprocess.run(cmd, check=False)
print("Return code:", result.returncode)
print("\nExpected run log dir:", RUN_LOG_DIR)
print("The following files should be there after training finishes:")
print("  fl_rounds.csv, fl_clients.csv, fl_eval.json, config.json, fl_summary.json, final_model.pth")


## Plot Results (accuracy, F1, losses, timings, confusion matrix)


In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

log_dir = RUN_LOG_DIR
rounds_path = os.path.join(log_dir, "fl_rounds.csv")
clients_path = os.path.join(log_dir, "fl_clients.csv")
eval_path = os.path.join(log_dir, "fl_eval.json")
summary_path = os.path.join(log_dir, "fl_summary.json")

if not os.path.isfile(rounds_path) or not os.path.isfile(clients_path):
    print("No results yet. Run the federated learning cell first.")
else:
    rounds_df = pd.read_csv(rounds_path)
    clients_df = pd.read_csv(clients_path)

    # Global test accuracy over rounds
    plt.figure(figsize=(10, 6))
    plt.plot(rounds_df["round"], rounds_df["global_test_acc"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Global Test Accuracy over FL Rounds (Colon Cancer)")
    plt.grid(True)
    plt.show()

    # F1 macro over rounds (if present)
    if "f1_macro" in rounds_df.columns:
        plt.figure(figsize=(10, 6))
        plt.plot(rounds_df["round"], rounds_df["f1_macro"], marker="o", linewidth=2, color="green")
        plt.xlabel("Round")
        plt.ylabel("F1 (macro)")
        plt.title("Global Test F1 (macro) over FL Rounds (Colon Cancer)")
        plt.grid(True)
        plt.show()

    # Inference and training time per round (if present)
    if "inference_time_sec" in rounds_df.columns and "training_time_sec" in rounds_df.columns:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(rounds_df["round"], rounds_df["inference_time_sec"], marker="o", label="Inference time (s)", linewidth=2)
        ax.plot(rounds_df["round"], rounds_df["training_time_sec"], marker="s", label="Training time (s)", linewidth=2)
        ax.set_xlabel("Round")
        ax.set_ylabel("Time (s)")
        ax.set_title("Inference & Training Time per Round (Colon Cancer)")
        ax.legend()
        ax.grid(True)
        plt.tight_layout()
        plt.show()

    # Client training accuracies
    plt.figure(figsize=(12, 6))
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_acc"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Accuracy (%)")
    plt.title("Client Training Accuracies over FL Rounds (Colon Cancer)")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Losses
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(rounds_df["round"], rounds_df["global_test_loss"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Loss")
    plt.title("Global Test Loss (Colon Cancer)")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_loss"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Loss")
    plt.title("Client Training Losses (Colon Cancer)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Final confusion matrix (from fl_eval.json, last round)
    if os.path.isfile(eval_path):
        with open(eval_path) as f:
            ev = json.load(f)
        rounds_ev = ev.get("rounds", [])
        class_names = ev.get("class_names")
        if not class_names and rounds_ev and "confusion_matrix" in rounds_ev[-1]:
            # Fallback generic names if class names are missing
            cm_tmp = np.array(rounds_ev[-1]["confusion_matrix"])
            class_names = [f"C{i}" for i in range(cm_tmp.shape[0])]

        if rounds_ev and "confusion_matrix" in rounds_ev[-1]:
            cm = np.array(rounds_ev[-1]["confusion_matrix"])
            plt.figure(figsize=(8, 6))
            plt.imshow(cm, interpolation="nearest", cmap="Blues")
            plt.colorbar()
            if class_names is not None:
                plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
                plt.yticks(np.arange(len(class_names)), class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title("Confusion Matrix (final round, Colon Cancer)")
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, int(cm[i, j]), ha="center", va="center",
                             color="black" if cm[i, j] < cm.max() / 2 else "white")
            plt.tight_layout()
            plt.show()

    if os.path.isfile(summary_path):
        with open(summary_path) as f:
            s = json.load(f)
        print("Summary:", json.dumps(s, indent=2))


## Copy important result files (including trained model) back into the repo and push to GitHub

This cell copies only the **essential artifacts** from the run-specific directory into the repository under `results/`,
then stages, commits, and (optionally) pushes them to GitHub.

Artifacts copied:
- `fl_rounds.csv`
- `fl_clients.csv`
- `fl_eval.json`
- `config.json`
- `fl_summary.json`
- `final_model.pth` (final trained model weights, so you can re-use them without retraining)


In [ ]:
import os
import shutil
import subprocess

PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

# Run-specific log directory (source) and repo destination directory
SRC_DIR = RUN_LOG_DIR
DEST_DIR = os.path.join(
    PROJECT_DIR, "results", DATASET_NAME, f"{NUM_CLIENTS}_clients", EXPERIMENT_TAG
)

FILES = [
    "fl_rounds.csv",
    "fl_clients.csv",
    "fl_eval.json",
    "config.json",
    "fl_summary.json",
    "final_model.pth",
]

os.makedirs(DEST_DIR, exist_ok=True)
copied = []
for f in FILES:
    src = os.path.join(SRC_DIR, f)
    dst = os.path.join(DEST_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        copied.append(os.path.relpath(dst, PROJECT_DIR))
    else:
        print(f"Skipping {f} (not found in {SRC_DIR})")

if not copied:
    print("No result files to push. Run the FL cell first.")
else:
    print("Copied to repo (under results/):")
    for rel in copied:
        print("  ", rel)

    # Stage copied files
    for rel in copied:
        subprocess.run(["git", "add", rel], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "status"], cwd=PROJECT_DIR, check=True)

    # Commit (only if there is something to commit)
    commit_res = subprocess.run(
        ["git", "commit", "-m", "Add colon cancer FL result artifacts"],
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
    )
    if commit_res.returncode == 0:
        print(commit_res.stdout)
    else:
        # 1 means "nothing to commit" for many git versions; print message and continue
        print(commit_res.stdout or commit_res.stderr)

    # Read token from environment (recommended: Colab Secrets -> env var)
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        print("\nNot pushing: missing GITHUB_TOKEN (Colab can't prompt for GitHub credentials).")
        print("Set GITHUB_TOKEN in the environment (e.g., Colab Secrets) and rerun this cell if you want to push.")
    else:
        origin_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT_DIR, text=True).strip()
        if origin_url.startswith("https://"):
            push_url = origin_url.replace("https://", f"https://{token}@")
            push_res = subprocess.run(["git", "push", push_url, "HEAD"], cwd=PROJECT_DIR)
        else:
            # SSH remote
            push_res = subprocess.run(["git", "push", "origin", "HEAD"], cwd=PROJECT_DIR)

        if push_res.returncode == 0:
            print("\nPushed to GitHub.")
        else:
            raise SystemExit("git push failed (see output above).")
